# Session 1: Tree-Based Classifiers, Survival Models & NER

In this session, we explore **tree-based classifiers** (Decision Trees, Random Forest, XGBoost), **survival models** (Kaplan-Meier, Cox Proportional Hazards), and **Named Entity Recognition** with spaCy and scispaCy.

---

## What You'll Learn

1. **Tree-Based Classifiers** - Decision Trees, Random Forest, XGBoost
2. **Survival Models** - Time-to-event prediction with Kaplan-Meier and Cox PH
3. **Named Entity Recognition** - Medical NER with spaCy and scispaCy

---

## Why Tree-Based Models?

Tree-based models are popular in healthcare ML because:
- **Interpretable**: Can visualize decision paths for clinicians
- **Handle missing data**: Many implementations handle NaN natively
- **Non-linear**: Capture complex feature interactions
- **No scaling required**: Don't need feature normalization

By the end of this session, you'll understand when to use each model type and have compared multiple approaches to patient risk prediction.

---

## Setup & Imports

We import libraries for:
- **Data manipulation**: pandas, numpy for data processing
- **Visualization**: matplotlib, seaborn for plots
- **Tree models**: sklearn's DecisionTreeClassifier, RandomForestClassifier, and xgboost
- **Survival analysis**: lifelines library for Kaplan-Meier and Cox PH models

These libraries are installed by the pip install cell below.

## Setting Environment Up for Colab

Mount Google Drive and set the repository path so this notebook can access the EHR data and source code.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

### Project Configuration

This notebook loads paths from a YAML config file instead of hard-coded paths.

**Setup (one-time):**
1. In Colab's left sidebar, click the **Key** icon (Secrets)
2. Add a secret named `PROJECT_CONFIG_PATH`
3. Set the value to your config file path (e.g., `/content/drive/MyDrive/Project/config.yaml`)
4. Toggle "Notebook access" ON

In [ ]:
from google.colab import userdata
import yaml

try:
    config_path = userdata.get('PROJECT_CONFIG_PATH')
except:
    config_path = input("Enter path to your config.yaml: ")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REPO_PATH = config['repo_path']
print(f"Repository path: {REPO_PATH}")

### Install Source Code as Package

`pip install` installs the local source code as a Python package so you can import directly from it (e.g., `from helpers import get_col`). Re-run this cell after making changes to the source code.

In [ ]:
!pip install {REPO_PATH}/src/ xgboost lifelines -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Tree-based models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Model evaluation
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

# Imputation
from sklearn.impute import SimpleImputer

# Survival analysis
from lifelines import KaplanMeierFitter, CoxPHFitter

print("Libraries imported")


In [ ]:
import os

DATA_DIR = os.path.join(REPO_PATH, 'data', 'week_2')
NOTES_DIR = os.path.join(REPO_PATH, 'data', 'week_1', 'processed_data')
OUTPUT_DIR = os.path.join(REPO_PATH, 'data', 'week_3')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"DATA_DIR:   {DATA_DIR}")
print(f"NOTES_DIR:  {NOTES_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

---

## Load Classifier Training Data

We load the classifier training data created in Week 2. This dataset contains:
- **Daily instances** for diabetic patients
- **Engineered features** (temporal, clinical, rolling windows)
- **Target**: `will_have_high_risk_event_next_30d` (1 = high-risk event within 30 days)

Each row represents a patient on a specific date with features computed using only data available before that date (temporal safety).

In [ ]:
# Load classifier training data from Week 2
df = pd.read_csv(os.path.join(DATA_DIR, 'classifier_training_data_22_features.csv'), low_memory=False)

print(f"Loaded {len(df):,} instances from {df['patient_id'].nunique():,} patients")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

### Identify Feature and Target Columns

We separate the columns into:
- **Feature columns**: All columns except patient ID, date, and target (used for prediction)
- **Target column**: `will_have_high_risk_event_next_30d` - what we're trying to predict
- **Patient column**: For patient-level splitting to prevent data leakage

We also examine the class distribution. High-risk events are typically rare (<5%), which creates a class imbalance challenge.

In [ ]:
# Identify feature columns
exclude_cols = ['patient_id', 'date']
exclude_patterns = ['label', 'risk', 'target', 'will_have', 'next', 'survival']
feature_cols = [col for col in df.columns
                if col not in exclude_cols
                and not any(pat in col.lower() for pat in exclude_patterns)]

label_col = 'will_have_high_risk_event_next_30d'
patient_col = 'patient_id'

print(f"Features ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  {col}: {df[col].dtype}")
print(f"\nTarget: {label_col}")
print(f"Positive rate: {df[label_col].mean():.2%}")

---

# Part 1: Tree-Based Classifiers

We'll explore three tree-based models, each with different strengths:

| Model | How It Works | Pros | Cons |
|-------|--------------|------|------|
| **Decision Tree** | Single tree with binary splits | Highly interpretable, shows decision path | Prone to overfitting |
| **Random Forest** | Ensemble of trees (bagging) | Robust, reduces variance | Less interpretable |
| **XGBoost** | Sequential trees (boosting) | Often best performance | More complex to tune |

We'll train each model and compare their performance on patient risk prediction.

## 1.1 Prepare Training Data

We use the **full dataset** for training (all daily instances per patient). This preserves the full class distribution and gives the models maximum data to learn from.

**Important**: We perform a **patient-level train/test split** using `GroupShuffleSplit` **before** any imputation. This ensures:
- The same patient never appears in both train and test sets
- Imputer medians are computed from training data only (no preprocessing leakage)
- We evaluate generalization to NEW patients, not just new time points

**Imputation Strategy — Why NOT `fillna(0)`?**

Using `fillna(0)` creates clinically impossible values: 47.8% of eGFR and 9.9% of HbA1c values become zero, which no living patient would have. Instead, we use a **dual strategy**:

| Model Type | Strategy | Rationale |
|------------|----------|-----------|
| **XGBoost** | NaN passthrough | XGBoost handles NaN natively — it learns optimal split directions for missing values |
| **Decision Tree, Random Forest** | Median imputation (train-only) | sklearn trees cannot handle NaN, so we impute with median from the training set |

This gives XGBoost the advantage of its native missing-value handling while keeping sklearn models functional.

In [ ]:
# Use full dataset for better class representation (not one-per-patient)
# With only ~3% positive rate, one-per-patient gives too few positives

print(f"Full dataset: {len(df):,} instances, {df['patient_id'].nunique():,} patients")
print(f"Positive rate: {df[label_col].mean()*100:.2f}%")

# Prepare features and labels
X_raw = df[feature_cols].copy()  # NaN passthrough for XGBoost
y = df[label_col].astype(int)
groups = df[patient_col]

# Patient-level split FIRST (before any imputation to avoid leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_raw, y, groups=groups))

# Raw splits with NaN passthrough (for XGBoost)
X_train_raw, X_test_raw = X_raw.iloc[train_idx], X_raw.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Median imputation for sklearn models (DT, RF) — fit on TRAIN ONLY to prevent leakage
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=feature_cols,
    index=X_train_raw.index
)
X_test = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=feature_cols,
    index=X_test_raw.index
)

print(f"\nImputation summary (medians from train set only):")
n_missing = X_raw.isna().sum()
for col in feature_cols:
    if n_missing[col] > 0:
        print(f"  {col}: {n_missing[col]:,} NaN → median={imputer.statistics_[feature_cols.index(col)]:.1f}")

print(f"\nTrain: {len(X_train):,} instances from {groups.iloc[train_idx].nunique():,} patients")
print(f"Test: {len(X_test):,} instances from {groups.iloc[test_idx].nunique():,} patients")
print(f"\nTrain positives: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Test positives: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

## 1.2 Decision Tree

A Decision Tree splits data recursively based on feature thresholds. At each node, it finds the feature and threshold that best separates the classes (maximizes information gain or Gini reduction).

**Key hyperparameters we set:**
- `max_depth=5`: Limit tree depth to prevent overfitting (deeper trees memorize training data)
- `min_samples_leaf=20`: Require at least 20 samples in each leaf for stable predictions
- `class_weight='balanced'`: Automatically weight classes inversely to their frequency (handles imbalance)

**Why use Decision Trees in healthcare?**
- Clinicians can follow the exact decision path
- Easy to explain WHY a patient was flagged as high-risk
- Aligns well with clinical decision rules

---

<details>
<summary><strong>Hint 1 — Key parameters</strong> (click to expand)</summary>

`DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, class_weight='balanced', random_state=42)`

Use `model.predict_proba(X_test)[:, 1]` to get positive-class probabilities.
</details>

<details>
<summary><strong>Hint 2 — Evaluation</strong> (click to expand)</summary>

After fitting, evaluate with `roc_auc_score(y_test, y_pred_proba)` and `average_precision_score(y_test, y_pred_proba)`. These metrics were introduced in Week 2 Session 3.
</details>

In [ ]:
# Train a Decision Tree
# TODO: Create a DecisionTreeClassifier with appropriate hyperparameters
#   - max_depth: limit tree depth to prevent overfitting
#   - min_samples_leaf: minimum samples at leaf nodes
#   - class_weight: handle class imbalance
#   - random_state=42 for reproducibility
dt = None

# TODO: Fit the model on training data
# dt.fit(...)

# TODO: Evaluate — get predicted probabilities and compute metrics
y_pred_proba = None
auc = None
ap = None

print(f"Decision Tree Performance:")
print(f" ROC-AUC: {auc}")
print(f" PR-AUC: {ap}")

### Visualize the Decision Tree

One major advantage of Decision Trees is **interpretability**. We can visualize the tree to see exactly what rules the model learned.

Each node shows:
- **Split condition**: The feature and threshold (e.g., "HbA1c <= 8.5")
- **Samples**: Number of training samples at this node
- **Class distribution**: How many belong to each class
- **Predicted class**: The majority class at this node

This visualization can be shared with clinicians to validate that the model's logic makes clinical sense.

In [ ]:
# Visualize the tree (first 3 levels)
plt.figure(figsize=(20, 10))
plot_tree(dt, 

    feature_names=feature_cols,
    class_names=['No Risk', 'High Risk'],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8)
plt.title("Decision Tree (First 3 Levels)")
plt.tight_layout()
plt.show()

print("\n Key insight: Decision trees show exactly WHY a prediction was made")

## 1.3 Random Forest

Random Forest is an **ensemble of decision trees** that uses **bagging** (bootstrap aggregating):

1. **Bootstrap sampling**: Each tree is trained on a random sample (with replacement) of the training data
2. **Feature randomization**: At each split, only a random subset of features is considered
3. **Averaging**: Final prediction is the average of all tree predictions

**Why this works:**
- Individual trees overfit to different patterns (high variance)
- Averaging reduces variance without increasing bias
- Result: More robust predictions than a single tree

**Key hyperparameters:**
- `n_estimators=100`: Number of trees (more = better but slower)
- `max_depth=10`: Maximum depth per tree
- `class_weight='balanced'`: Handle class imbalance

---

<details>
<summary><strong>Hint 1 — Key parameters</strong> (click to expand)</summary>

`RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=10, class_weight='balanced', random_state=42, n_jobs=-1)`

More trees (`n_estimators`) generally helps up to a point. `n_jobs=-1` uses all CPU cores.
</details>

<details>
<summary><strong>Hint 2 — Evaluation</strong> (click to expand)</summary>

Same as Decision Tree: `predict_proba(X_test)[:, 1]` then `roc_auc_score` and `average_precision_score`. Use the imputed `X_train`/`X_test` (Random Forest doesn't handle NaN natively).
</details>

In [ ]:
# Train Random Forest
# TODO: Create a RandomForestClassifier
#   - n_estimators: number of trees in the forest
#   - max_depth: maximum depth per tree
#   - min_samples_leaf: minimum samples at leaf nodes
#   - class_weight='balanced' for imbalanced data
#   - random_state=42, n_jobs=-1
rf = None

# TODO: Fit the model on training data
# rf.fit(...)

# TODO: Evaluate — get predicted probabilities and compute metrics
y_pred_proba_rf = None
auc_rf = None
ap_rf = None

print(f"Random Forest Performance:")
print(f" ROC-AUC: {auc_rf}")
print(f" PR-AUC: {ap_rf}")

### Random Forest Feature Importance

While we can't visualize the entire forest (100 trees), we can extract **feature importance scores**. These tell us which features the model relies on most.

**How it's calculated (Mean Decrease in Impurity / Gini importance):**
- For each feature, measure the **total reduction in impurity (Gini)** across all splits using that feature
- Average across all trees in the forest
- Higher importance = greater total impurity reduction (not just split count — that would be the "weight" metric)

**Clinical validation:**
- Important features should make clinical sense (e.g., HbA1c for diabetes risk)
- If an unexpected feature dominates, investigate for possible data leakage

In [ ]:
# Feature importance from Random Forest
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
top_n = 15
plt.barh(range(top_n), importance_df['importance'].head(top_n), color='forestgreen')
plt.yticks(range(top_n), importance_df['feature'].head(top_n))
plt.xlabel('Importance')
plt.title('Random Forest Feature Importance (Top 15)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 1.4 XGBoost

XGBoost (eXtreme Gradient Boosting) builds trees **sequentially** using **boosting**:

1. Train first tree on the original data
2. Calculate residuals (errors) from tree 1
3. Train tree 2 to predict the residuals (correct errors)
4. Add tree 2's predictions to tree 1's (with a learning rate)
5. Repeat until `n_estimators` trees

**Key hyperparameters:**
- `learning_rate`: How much each tree contributes (smaller = more trees needed but better generalization)
- `max_depth`: Typically shallower than Random Forest (3-10)
- `scale_pos_weight`: Ratio of negative to positive samples (handles imbalance)

**Advantages:**
- Often achieves best performance on structured/tabular data
- Handles missing values natively
- Built-in L1/L2 regularization prevents overfitting

In [ ]:
# Calculate scale_pos_weight for imbalanced data
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / max(pos_count, 1)

print(f"Class imbalance: {neg_count:,} negative, {pos_count:,} positive")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

### Train XGBoost Model

We train XGBoost with:
- `scale_pos_weight`: Calculated above to handle class imbalance
- `eval_metric='aucpr'`: Optimize for PR-AUC (better than ROC-AUC for imbalanced data)
- Moderate hyperparameters that we'll tune in Session 2

---

<details>
<summary><strong>Hint 1 — Key parameters</strong> (click to expand)</summary>

`xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='aucpr')`

XGBoost handles NaN natively — use `X_train_raw` / `X_test_raw`, NOT the imputed versions.
</details>

<details>
<summary><strong>Hint 2 — Why NaN passthrough matters</strong> (click to expand)</summary>

XGBoost learns an optimal direction for missing values at each split. Imputing with median would destroy this: a patient with missing eGFR (never tested) is clinically different from a patient with median eGFR. Using raw NaN lets the model learn this distinction.
</details>

In [ ]:
# Train XGBoost — uses raw data with NaN passthrough (not median-imputed)
# TODO: Create an XGBClassifier
#   - n_estimators, max_depth, learning_rate: tree structure
#   - scale_pos_weight: from cell above (handles class imbalance)
#   - eval_metric='aucpr': optimize for PR-AUC
#   - random_state=42
xgb_model = None

# TODO: Fit on X_train_raw (not imputed — XGBoost handles NaN natively)
# xgb_model.fit(...)

# TODO: Evaluate on X_test_raw
y_pred_proba_xgb = None
auc_xgb = None
ap_xgb = None

print(f"XGBoost Performance (NaN passthrough):")
print(f" ROC-AUC: {auc_xgb}")
print(f" PR-AUC: {ap_xgb}")

## 1.5 Model Comparison

Let's compare all three tree-based models side by side.

**Metrics:**
- **ROC-AUC**: Area under ROC curve (0.5 = random, 1.0 = perfect). Measures discrimination ability.
- **PR-AUC**: Area under Precision-Recall curve. More informative for imbalanced data because it focuses on the positive class.

**Expected results:**
- XGBoost often performs best (but not always)
- Random Forest is more robust than single Decision Tree
- All models may show poor metrics if data is very imbalanced or sample size is small

In [ ]:
# Compare all three models
print("MODEL COMPARISON")
print("=" * 50)
print(f"{'Model':<20} {'ROC-AUC':>10} {'PR-AUC':>10}")
print("-" * 50)
print(f"{'Decision Tree':<20} {auc:>10.4f} {ap:>10.4f}")
print(f"{'Random Forest':<20} {auc_rf:>10.4f} {ap_rf:>10.4f}")
print(f"{'XGBoost':<20} {auc_xgb:>10.4f} {ap_xgb:>10.4f}")
print("=" * 50)

## 1.6 Progress from Week 2 Baselines

How much have we improved over the simple baselines from Week 2? The Week 2 models used only 6 features; we now have 22 engineered features plus non-linear models.

| Model | Type | Features | PR-AUC | Source |
|-------|------|----------|--------|--------|
| HbA1c > 9 rule | Rule-based (Week 2) | 1 | ~0.155 | Week 2 Session 2 |
| Logistic Regression | Linear (Week 2) | 6 | 0.225 | Week 2 Session 3 |
| XGBoost | Tree ensemble (Week 3) | 22 | ~0.878 | This session |

The improvement comes from two sources: more features (6 → 22) and non-linear models that capture complex interactions. Note that the Week 2 LR used only 6 basic features — the comparison shows the combined effect of better features AND better models.

In [ ]:
# Visualize progression from Week 2 baselines to Week 3 models
# Week 2 values from session_2_baselines.ipynb and session_3_evaluation_metrics.ipynb
week2_baselines = {
    'HbA1c > 9\n(Rule, 1 feat)': 0.155,
    'Logistic Reg\n(Week 2, 6 feat)': 0.225,
}
week3_models = {
    'Decision Tree': ap,
    'Random Forest': ap_rf,
    'XGBoost': ap_xgb,
}

fig, ax = plt.subplots(figsize=(10, 6))

# Week 2 bars
x_w2 = range(len(week2_baselines))
bars_w2 = ax.bar(x_w2, week2_baselines.values(), color='#cccccc', edgecolor='gray', width=0.6, label='Week 2 Baselines')

# Week 3 bars
x_w3 = range(len(week2_baselines), len(week2_baselines) + len(week3_models))
bars_w3 = ax.bar(x_w3, week3_models.values(), color='steelblue', edgecolor='navy', width=0.6, label='Week 3 Models (22 features)')

# Labels
all_labels = list(week2_baselines.keys()) + list(week3_models.keys())
ax.set_xticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, fontsize=9)

# Value annotations
for bar in list(bars_w2) + list(bars_w3):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('PR-AUC')
ax.set_title('Model Progression: Week 2 Baselines → Week 3 Advanced Models')
ax.legend()
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

# Part 2: Survival Models

Survival analysis models **time-to-event** data. Instead of predicting IF an event happens (binary classification), we model WHEN it happens.

**Key concepts:**
- **Event**: The outcome we're predicting (hospitalization, death, disease progression)
- **Duration**: Time from observation start until event or end of follow-up
- **Censoring**: When we don't observe the event (patient leaves study, study ends)

**Why survival analysis for patient risk?**
- More informative than binary classification (captures WHEN, not just IF)
- Handles variable follow-up times naturally
- Can estimate risk at any future time point
- Standard methodology in clinical trials and medical research

## 2.1 Prepare Survival Data

The classifier training data already includes survival analysis columns:

- **`survival_days_until_event`**: Days from this date until the next high-risk event
- **`survival_event_observed`**: 1 if event occurred, 0 if censored (no event during observation)

These were computed during dataset generation by looking forward from each observation date to find the next:
- Emergency visit
- Hospitalization
- HbA1c >= 10%

For survival models, we'll sample ONE observation per patient (their first observation) to create a standard survival dataset.

In [ ]:
# Use survival columns from the dataset (already computed during generation)
# Sample ONE observation per patient (first observation)
df['date'] = pd.to_datetime(df['date'])
df_sorted = df.sort_values(['patient_id', 'date'])
survival_df = df_sorted.groupby('patient_id').first().reset_index()

# Rename columns to match lifelines convention
survival_df['duration'] = survival_df['survival_days_until_event']
survival_df['event'] = survival_df['survival_event_observed']

print(f"Survival data from dataset:")
print(f" Patients: {len(survival_df):,}")
print(f" Events observed: {survival_df['event'].sum():,} ({survival_df['event'].mean()*100:.1f}%)")
print(f" Censored: {(survival_df['event'] == 0).sum():,} ({(1-survival_df['event'].mean())*100:.1f}%)")
print(f" Duration range: {survival_df['duration'].min():.0f} - {survival_df['duration'].max():.0f} days")
print(f" Mean duration: {survival_df['duration'].mean():.1f} days")

## 2.2 Kaplan-Meier Survival Curves

The **Kaplan-Meier estimator** is the most common non-parametric method for estimating survival probability over time.

**How to read the plot:**
- **X-axis**: Time (days since first observation)
- **Y-axis**: Probability of remaining event-free until that time
- **Steps**: Curve drops at each event time
- **Shaded area**: 95% confidence interval

**Interpretation:**
- At day 0, survival probability = 1.0 (everyone event-free)
- Curve decreases as high-risk events occur
- Steeper drop = higher risk during that period
- Plateau = stable period with few events

---

<details>
<summary><strong>Hint 1 — Fitting KM</strong> (click to expand)</summary>

`kmf.fit(durations, event_observed=events)` — that's the full API. For plotting: `kmf.plot_survival_function()`.
</details>

<details>
<summary><strong>Hint 2 — Survival probabilities</strong> (click to expand)</summary>

Use `kmf.predict(days)` to get the probability of remaining event-free at a specific time point. Loop through `[30, 90, 180, 365, 730]`.
</details>

In [ ]:
# Fit Kaplan-Meier model
# TODO: Create a KaplanMeierFitter and fit it
#   - Use survival_df['duration'] for durations
#   - Use survival_df['event'] for event_observed
kmf = KaplanMeierFitter()
# kmf.fit(...)

# TODO: Plot the survival curve
#   - Use kmf.plot_survival_function()
#   - Add title, axis labels, grid
plt.figure(figsize=(10, 6))
# Your plotting code here
plt.show()

# TODO: Print survival probabilities at key time points (30, 90, 180, 365, 730 days)
#   - Use kmf.predict(days) to get probability at each time point
print("Event-free probability at key time points:")
for days in [30, 90, 180, 365, 730]:
    prob = None  # TODO: use kmf.predict()
    print(f"  {days:>3} days: {prob}")

### Stratified Kaplan-Meier Curves

We can compare survival curves between groups to identify risk factors. This is useful for:
- Comparing high-risk vs low-risk patients
- Evaluating treatment effects
- Validating that clinical risk factors affect outcomes

Here we compare patients with **high vs low HbA1c** levels (split at median). If HbA1c is a risk factor, we expect:
- High HbA1c patients to have **lower survival** (faster curve decline)
- The curves to **separate** over time

---

<details>
<summary><strong>Hint 1 — Stratification approach</strong> (click to expand)</summary>

1. Filter out NaN: `valid = survival_df[survival_df[col].notna()]`
2. Split at median: `high = valid[valid[col] >= median]`
3. Fit two separate KaplanMeierFitter objects and plot both
</details>

<details>
<summary><strong>Hint 2 — Log-rank test</strong> (click to expand)</summary>

`logrank_test(durations_A, durations_B, event_observed_A=events_A, event_observed_B=events_B)` — returns a result object with `.p_value`. If p < 0.05, the difference is statistically significant.
</details>

In [ ]:
# Compare survival curves by HbA1c level
from lifelines.statistics import logrank_test

hba1c_col = 'current_hba1c_level'

# TODO: Split patients into high vs low HbA1c groups
#   1. Filter to patients with non-NaN HbA1c
#   2. Compute median HbA1c
#   3. Split into high (>= median) and low (< median)
valid_hba1c = None
median_hba1c = None
high_hba1c = None
low_hba1c = None

# TODO: Fit KM curves for each group and plot them
plt.figure(figsize=(10, 6))

# Fit and plot high HbA1c group
kmf_high = KaplanMeierFitter()
# kmf_high.fit(..., label=f'High HbA1c (>={median_hba1c:.1f}%)')
# kmf_high.plot_survival_function()

# Fit and plot low HbA1c group
kmf_low = KaplanMeierFitter()
# kmf_low.fit(..., label=f'Low HbA1c (<{median_hba1c:.1f}%)')
# kmf_low.plot_survival_function()

plt.title('Survival Curves by HbA1c Level')
plt.xlabel('Days Since First Observation')
plt.ylabel('Event-Free Probability')
plt.grid(True, alpha=0.3)
plt.show()

# TODO: Run log-rank test for statistical significance
#   - Use logrank_test() with durations and events from both groups
# results = logrank_test(...)
# print(f"Log-rank test p-value: {results.p_value:.4f}")

## 2.3 Cox Proportional Hazards Model

Cox PH is the **regression equivalent** of survival analysis. It models how features affect the **hazard rate** (instantaneous risk of event at time t, given survival until t).

**The model equation:**
```
h(t|X) = h₀(t) × exp(β₁X₁ + β₂X₂ + ... + βₙXₙ)
```

Where:
- `h(t|X)`: Hazard at time t given features X
- `h₀(t)`: Baseline hazard (no assumption about shape - "semi-parametric")
- `βᵢ`: Coefficient for feature i
- `exp(βᵢ)`: **Hazard ratio** for feature i

**Hazard ratio interpretation:**
- HR = 1: No effect on risk
- HR > 1: Increased risk (e.g., HR=2 means 2× the risk)
- HR < 1: Decreased risk (protective factor)

We use a penalized model (`penalizer=0.1`) to handle potential multicollinearity between features.

---

<details>
<summary><strong>Hint 1 — Data preparation</strong> (click to expand)</summary>

`cox_df = survival_df[['duration', 'event'] + selected_features].copy()` then `cox_df = cox_df.fillna(cox_df.median())`. Cox PH cannot handle NaN (unlike XGBoost). Note: this exploratory fit uses the full dataset median — for proper evaluation with train/test split, see Section 2.4.
</details>

<details>
<summary><strong>Hint 2 — Fitting Cox PH</strong> (click to expand)</summary>

`cph = CoxPHFitter(penalizer=0.1)` then `cph.fit(cox_df, duration_col='duration', event_col='event')`. The penalizer prevents overfitting. Print summary with `cph.print_summary()`.
</details>

In [ ]:
# Prepare data for Cox PH - select clinically meaningful features
selected_features = ['current_hba1c_level', 'age_at_date', 'encounters_last_90d', 
    'days_since_last_hba1c', 'current_systolic_bp', 'bmi_category']

# TODO: Prepare the Cox PH dataset
#   1. Select duration, event, and selected_features columns from survival_df
#   2. Fill NaN with median values (exploratory fit — Section 2.4 uses train-only medians)
cox_df = None

# TODO: Fit a CoxPHFitter with penalizer=0.1
#   - Use duration_col='duration' and event_col='event'
cph = CoxPHFitter(penalizer=0.1)
# cph.fit(...)

# TODO: Print the model summary
# cph.print_summary(decimals=3, columns=['coef', 'exp(coef)', 'p'])

### Visualize Cox PH Coefficients

We can visualize the model coefficients as a **forest plot** to see which features increase or decrease risk.

**Reading the forest plot:**
- **Vertical line at 0**: No effect (reference line for log-hazard ratios)
- **Points to the right** (coef > 0): Increased risk
- **Points to the left** (coef < 0): Decreased risk (protective)
- **Error bars**: 95% confidence interval (if it crosses 0, effect is not statistically significant)

Note: `cph.plot()` displays **log-hazard ratios** (coefficients), not hazard ratios. The reference line is at 0, not 1.0. To interpret as hazard ratios, exponentiate: exp(coef) > 1 means increased risk.

In [ ]:
# Plot log-hazard ratios / coefficients (forest plot)
plt.figure(figsize=(10, 6))
cph.plot()
plt.title('Cox PH Log-Hazard Ratios / Coefficients (95% CI)')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


**Coefficient Interpretation (log-hazard ratios):**

- `exp(coef) > 1`: Higher values increase risk
- `exp(coef) < 1`: Higher values decrease risk (protective)
- `exp(coef) = 1`: No effect

Example: A hazard ratio (HR) of 1.5 means 50% higher risk per unit increase.

## 2.4 Evaluate Cox PH on Test Set

Just like classification models, we should evaluate survival models on **held-out test data** to assess generalization. We use:

- **Patient-level split**: Same patient never in both train and test
- **C-index (Concordance Index)**: The survival equivalent of ROC-AUC

**C-index interpretation:**
- C-index = 0.5: Random predictions (no predictive ability)
- C-index = 1.0: Perfect predictions
- C-index > 0.7: Generally considered good for clinical models

In [ ]:
# Reuse the same patient-level split from the classifier section above
# This ensures consistent train/test patients across all model types
train_patients = groups.iloc[train_idx].unique()
test_patients = groups.iloc[test_idx].unique()

# Create train and test DataFrames using the same patient split
survival_train = survival_df[survival_df['patient_id'].isin(train_patients)].copy()
survival_test = survival_df[survival_df['patient_id'].isin(test_patients)].copy()

print(f"Train set: {len(survival_train)} patients, {survival_train['event'].sum()} events ({survival_train['event'].mean()*100:.1f}%)")
print(f"Test set: {len(survival_test)} patients, {survival_test['event'].sum()} events ({survival_test['event'].mean()*100:.1f}%)")
print(f"\nNote: Using same patient split as classifiers for fair comparison")

In [ ]:
# Prepare train and test data for Cox PH
cox_train = survival_train[['duration', 'event'] + selected_features].copy()
cox_test = survival_test[['duration', 'event'] + selected_features].copy()

# Fill NaN with training set median (important: use train median for both!)
train_medians = cox_train.median()
cox_train = cox_train.fillna(train_medians)
cox_test = cox_test.fillna(train_medians)

# Fit Cox PH on TRAINING data only
cph_eval = CoxPHFitter(penalizer=0.1)
cph_eval.fit(cox_train, duration_col='duration', event_col='event')

# Evaluate on both train and test
train_cindex = cph_eval.score(cox_train, scoring_method='concordance_index')
test_cindex = cph_eval.score(cox_test, scoring_method='concordance_index')

print("Cox PH Model Evaluation:")
print("=" * 40)
print(f"Train C-index: {train_cindex:.4f}")
print(f"Test C-index: {test_cindex:.4f}")
print(f"\nOverfit gap: {train_cindex - test_cindex:.4f}")

if test_cindex > 0.6:
    print("\n Model shows predictive ability on unseen patients")
else:
    print("\n Model has limited predictive ability on test set")

### Survival Model Summary

| Metric | Train | Test |
|--------|-------|------|
| C-index | ~0.58 | ~0.55 |

**Why is C-index so much lower than classifier AUROC (~0.95)?**

Cox PH and tree-based classifiers solve fundamentally different problems:

| Aspect | Cox PH (C-index ~0.55) | XGBoost (AUROC ~0.95) |
|--------|------------------------|------------------------|
| **Task** | Predict *when* event occurs (time-to-event) | Predict *if* event occurs in next 30 days (binary) |
| **Features used** | 6 hand-selected clinical features | All 22 engineered features |
| **Model assumptions** | Linear + proportional hazards | Non-linear, captures interactions |
| **Data used** | 1 row per patient (first observation) | All daily instances (~580K rows) |
| **Appropriate metric** | C-index (ranking concordance) | AUROC (classification discrimination) |

**Key takeaways:**
- The comparison is **not apples-to-apples** — different tasks, data, and feature sets
- Cox PH with only 6 features and 2,109 patients has far less signal to work with
- Survival models excel at **clinical inference** (hazard ratios tell us *how much* each factor increases risk) rather than pure prediction
- A C-index of 0.55-0.58 is modest but typical for survival models with limited covariates; adding more features or using tree-based survival models (e.g., Random Survival Forest) could improve it
- In practice, survival models and classifiers serve **complementary roles**: Cox PH for understanding risk factors, classifiers for operational alerting

---

# Part 3: Named Entity Recognition (NER)

In Week 1, you used regex patterns to extract data from clinical notes. Now we explore more advanced methods using **Named Entity Recognition (NER)** with spaCy and scispaCy.

NER models can automatically detect entities like medications, diagnoses, and chemicals from clinical text, scaling better than hand-crafted regex and handling more varied text.

## 3.1 Load Clinical Notes

Load the clinical notes provided for extraction. These notes contain information about encounters that were removed from the structured CSV files, requiring NLP to extract the data.

In [ ]:
# Load sample clinical notes (CSV format)
notes_path = os.path.join(NOTES_DIR, 'notes_for_extraction.csv')
notes_df = pd.read_csv(notes_path, nrows=5000)
notes = notes_df.to_dict('records')

print(f"Loaded {len(notes)} clinical notes from {notes_path}")
print(f"Columns: {list(notes_df.columns)}")
print(f"\nSample note (first 200 chars):")
if 'note_text' in notes_df.columns:
    sample_text = str(notes_df['note_text'].iloc[0])[:200]
    print(f"  {sample_text}...")
else:
    text_cols = [c for c in notes_df.columns if notes_df[c].dtype == 'object' and c != 'patient_id']
    if text_cols:
        sample_text = str(notes_df[text_cols[0]].iloc[0])[:200]
        print(f"  {sample_text}...")

## 3.2 Named Entity Recognition with spaCy

spaCy is a fast, production-ready NLP library. The base English model (`en_core_web_sm`) recognizes general entities like:
- PERSON, ORG, GPE (locations)
- DATE, TIME, MONEY, QUANTITY

For medical text, specialized models like **scispaCy** or **medspaCy** can recognize:
- Diseases and conditions (DISEASE)
- Medications (MEDICATION)
- Anatomical structures (ANATOMY)
- Medical procedures (PROCEDURE)

We'll start with the base model to see what it finds.

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
# Load spaCy model
import spacy

nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded")

### Extract Named Entities from a Sample Note

Let's process a clinical note and see what entities spaCy finds. While the base model isn't trained on medical text, it can still identify useful entities like dates and organizations (hospitals, clinics).

---

<details>
<summary><strong>Hint 1 — spaCy processing</strong> (click to expand)</summary>

`doc = nlp(text[:2000])` — limit text length to avoid slow processing. Entities are in `doc.ents`. Each entity has `.text` and `.label_`.
</details>

In [ ]:
# Extract entities from a sample note
sample_text = notes[0]['note_text']

# TODO: Process the text with spaCy
#   1. Run nlp() on the first 2000 chars of sample_text
#   2. Loop through doc.ents and print label + text
doc = None

print("Named Entities Found:")
print("=" * 50)
# TODO: Print first 20 entities with their labels
# for ent in doc.ents[:20]:
#     print(f" {ent.label_:10} | {ent.text}")

## 3.3 Medical NER with scispaCy

The base spaCy model doesn't recognize medical entities. **scispaCy** is trained on biomedical text and can detect:
- **ENTITY**: Diseases, chemicals, genes, cell types
- Works with UMLS linking for standardized medical codes

| Model | Size | What it detects |
|-------|------|-----------------|
| `en_core_sci_sm` | Small | General scientific entities |
| `en_core_sci_md` | Medium | Better accuracy, includes word vectors |
| `en_ner_bc5cdr_md` | Medium | Diseases and chemicals specifically |

Install: `pip install scispacy` then `pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz`


In [ ]:
# Install scispaCy model (run once)
!pip install scispacy -q
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz -q

print("scispaCy and medical NER model installed")

In [ ]:
# Load scispaCy medical NER model
import scispacy
nlp_med = spacy.load("en_ner_bc5cdr_md")

print("scispaCy medical NER model loaded")
print("Model: en_ner_bc5cdr_md (diseases and chemicals)")

### Extract Medical Entities from Clinical Notes

Now we use scispaCy to extract diseases and chemicals (medications) from clinical notes. This is much more useful than the base spaCy model for healthcare data.

---

<details>
<summary><strong>Hint 1 — scispaCy entity types</strong> (click to expand)</summary>

The `en_ner_bc5cdr_md` model recognizes two entity types: `"DISEASE"` and `"CHEMICAL"`. Loop through `doc_med.ents`, check `ent.label_`, and append to the appropriate list. Use `set()` to deduplicate.
</details>

In [ ]:
# Extract medical entities from a sample note
sample_text = notes[0]['note_text']

# TODO: Process the text with the medical NER model (nlp_med)
#   1. Run nlp_med() on the first 3000 chars
#   2. Separate entities into diseases and chemicals lists
#   3. Deduplicate with set()
doc_med = None
diseases = []
chemicals = []

# TODO: Loop through doc_med.ents and sort by label
# for ent in doc_med.ents:
#     if ent.label_ == "DISEASE": ...
#     elif ent.label_ == "CHEMICAL": ...

# Deduplicate
diseases = list(set(diseases))
chemicals = list(set(chemicals))

print("Medical Entities Extracted by scispaCy:")
print("=" * 50)
print(f"\nDISEASES ({len(diseases)}):")
for d in diseases[:10]:
    print(f"  - {d}")

print(f"\nCHEMICALS/MEDICATIONS ({len(chemicals)}):")
for c in chemicals[:10]:
    print(f"  - {c}")

---
## Summary: Session Complete!

### What You Accomplished

1. **Tree-Based Classifiers**
   - Decision Trees: Highly interpretable, can visualize decision paths
   - Random Forest: Ensemble of trees, more robust than single tree
   - XGBoost: Gradient boosting, often achieves best performance

2. **Survival Models**
   - Kaplan-Meier: Visualize survival probability over time
   - Cox PH: Regression for time-to-event, outputs hazard ratios

3. **Named Entity Recognition**
   - Base spaCy NER for general entity extraction
   - scispaCy medical NER for diseases and medications

### Next Steps

In **Session 2**, you'll learn hyperparameter tuning, class imbalance handling, and ensemble methods.

### Professional Tip

When presenting models to clinicians, start with interpretable models (Decision Trees) to build trust. Then show that ensemble models achieve better performance while explaining they are "many trees voting together."